# PyCAM-SIMA persistent Dask and checkpoint fan-out

This standalone Notebook demonstrates both Dask control modes. Checkpoint tasks create restartable 24-rank MPI segments and fan out independent branches. A persistent Dask Actor instead owns one authenticated `NotebookSession`, starts MPI once, and reuses the same live StatePool for asynchronous phase, scheme, step, and field commands. In PBS mode, `fork_persistent()` can also keep the base snapshot in Dask memory and restore several independent, long-lived MPI children without checkpoint files. Inside a one-node allocation, ordinary segments and one persistent model avoid nested PBS jobs; persistent multi-model fan-out intentionally requires separate PBS jobs. All paths use the same manifest-driven `DeviceRegistry` to connect Python-owned fields to unchanged original Fortran schemes.

## 1. Execution model

```text
Jupyter/PBS controller + local Dask Client
           │
           ├── base MPI segment: 24 ranks × 10 steps
           │                    │
           │                    └── immutable checkpoint Future
           │                                  │
           ├──────────────────────────────────┼── control:    5 steps
           ├──────────────────────────────────└── no-kessler: 5 steps
           └── warm-initial: +1 K, 0 steps
                         │ exact edit check
                         └── warm: 5 steps
```

For the checkpoint fan-out section, the base process exits after writing its checkpoint. Each segment restores private NumPy arrays and creates a new `MPI.COMM_WORLD`; this is checkpoint/restart fan-out, not operating-system `fork()`. The persistent sections change the lifetime: one Actor keeps the same 24 MPI ranks alive across many method calls, while `fork_persistent()` transfers an immutable in-memory snapshot into several independent child Actors. With `execution_mode='allocation'`, segment and single-Actor paths remain in the same outer `PBS_JOBID`; persistent multi-model fork is skipped because every child needs its own 24-rank PBS resource.

## 2. Configure one experiment

Run this cell again to create a fresh timestamp before repeating the experiment. Existing branch directories are deliberately never overwritten.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from netCDF4 import Dataset
from pycam_sima import (
    BranchSpec,
    DaskExperimentClient,
    FieldEdit,
    ObserveFields,
    RunPhase,
    RunScheme,
    SegmentPlan,
)

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/dask_notebook_trials' / f'fanout-{stamp}'
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'branches'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
dask_workers = 1 if execution_mode == 'allocation' else 3

print('pycam_sima', pycam_sima.__version__)
print('experiment root:', experiment_root)
print('execution mode:', execution_mode)

pycam_sima 0.11.0
experiment root: /glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516
execution mode: allocation


## 3. Create the Dask controller

The Dask workers orchestrate 24-rank MPI segments. In `allocation` mode one worker serially launches every segment with `mpiexec` inside the current PBS job. In `pbs` mode three workers may independently submit branch PBS jobs.

In [3]:
if 'client' in globals():
    client.close()

client = Client(
    processes=False,
    n_workers=dask_workers,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://10.14.9.97:8787/status,
Dashboard: http://10.14.9.97:8787/status,Workers: 1
Total threads: 1,Total memory: 80.00 GiB
Status: running,Using processes: False
Comm: inproc://10.14.9.97/24617/1,Workers: 0
Dashboard: http://10.14.9.97:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://10.14.9.97/24617/4,Total threads: 1
Dashboard: http://10.14.9.97:43125/status,Memory: 80.00 GiB
Nanny: None,


## 4. Submit the common base and the zero-step warm edit

This first stage runs two Dask-managed MPI segments. The base runs 10 steps. `warm-initial` restores that exact state, adds `1 K` to `air_temperature`, runs zero model steps, and writes another checkpoint. In allocation mode neither segment submits another PBS job.

In [4]:
base = experiments.submit_base(
    BranchSpec('base', steps=10)
)

warm_initial = experiments.submit_branch(
    base,
    BranchSpec(
        'warm-initial',
        steps=0,
        field_edits=(
            FieldEdit('air_temperature', 'add', 1.0),
        ),
    ),
)

initial_summaries = experiments.summaries({
    'base': base,
    'warm-initial': warm_initial,
})
initial_summaries

{'base': {'branch': 'base',
  'parent_branch': None,
  'step': 10,
  'history_samples': 11,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/base/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/base/history',
  'checkpoint_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/base/checkpoint',
  'snapshot_nbytes': 37660986,
  'execution_mode': 'allocation',
  'pbs_job_id': '6874956.desched1',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_base_04b28bdb44bc.log',
  'action_count': 1,
  'action_trace': [{'action': {'count': 10, 'type': 'run_steps'},
    'index': 0,
    'last_phase': 'physics_timestep_initial',
    'last_scheme': 'physics_before_coupler.kessler_diagnostics',
    'last_scheme_group': 'physics_before_coupler',
    'native_calls_delta': 2984,
    'step_after': 10,
    'step

## 5. Verify the edit before model evolution

Both checkpoints are at model step 10. This comparison therefore tests only `FieldEdit('air_temperature', 'add', 1.0)` across all 24 ranks. Every edited array must match NumPy's elementwise `base + 1.0` operation bit for bit. Recomputing `(base + 1.0) - base` performs another floating-point operation, so that diagnostic is required to equal `1 K` only within one spacing of the largest base value.

In [5]:
def checkpoint_field(summary_map, branch, field, rank=0):
    checkpoint_file = (
        Path(summary_map[branch]['checkpoint_dir'])
        / f'rank-{rank:03d}.npz'
    )
    with np.load(checkpoint_file, allow_pickle=False) as arrays:
        return arrays[field].copy()

base_checkpoint = Path(initial_summaries['base']['checkpoint_dir'])
rank_count = len(tuple(base_checkpoint.glob('rank-*.npz')))
base_temperatures = [
    checkpoint_field(initial_summaries, 'base', 'air_temperature', rank)
    for rank in range(rank_count)
]
warm_initial_temperatures = [
    checkpoint_field(
        initial_summaries, 'warm-initial', 'air_temperature', rank
    )
    for rank in range(rank_count)
]
initial_differences = [
    warm - base
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
]
exact_numpy_edit = all(
    np.array_equal(warm, np.add(base, 1.0))
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
)
maximum_roundoff_from_1K = float(
    max(np.abs(delta - 1.0).max() for delta in initial_differences)
)
roundoff_tolerance = float(
    max(np.spacing(np.abs(base).max()) for base in base_temperatures)
)
difference_within_roundoff = (
    maximum_roundoff_from_1K <= roundoff_tolerance
)

assert exact_numpy_edit
assert difference_within_roundoff
sample_count = sum(delta.size for delta in initial_differences)
{
    'ranks': rank_count,
    'rank_local_shape': initial_differences[0].shape,
    'difference_min': float(min(delta.min() for delta in initial_differences)),
    'difference_max': float(max(delta.max() for delta in initial_differences)),
    'difference_mean': float(
        sum(delta.sum() for delta in initial_differences) / sample_count
    ),
    'exact_numpy_add_1K': exact_numpy_edit,
    'maximum_roundoff_from_1K': maximum_roundoff_from_1K,
    'roundoff_tolerance': roundoff_tolerance,
    'difference_within_roundoff': difference_within_roundoff,
}

{'ranks': 24,
 'rank_local_shape': (4, 4, 30, 3, 3),
 'difference_min': 0.9999999999999716,
 'difference_max': 1.0000000000000284,
 'difference_mean': 1.0,
 'exact_numpy_add_1K': True,
 'maximum_roundoff_from_1K': 2.842170943040401e-14,
 'roundoff_tolerance': 5.684341886080802e-14,
 'difference_within_roundoff': True}

## 6. Continue the three five-step experiments

`control` and `no-kessler` continue directly from the base checkpoint. `warm` continues from the already verified `warm-initial` checkpoint without applying a second edit. This stage creates three additional Dask tasks; allocation mode executes their full-node MPI segments serially in the same PBS job.

In [6]:
branches = experiments.fork(
    base,
    (
        BranchSpec('control', steps=5),
        BranchSpec(
            'no-kessler',
            steps=5,
            disable_schemes=('kessler',),
        ),
    ),
)
branches['warm'] = experiments.submit_branch(
    warm_initial,
    BranchSpec('warm', steps=5),
)

summaries = experiments.summaries(branches)
summaries

{'control': {'branch': 'control',
  'parent_branch': 'base',
  'step': 15,
  'history_samples': 16,
  'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/control/run',
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/control/history',
  'checkpoint_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/control/checkpoint',
  'snapshot_nbytes': 37660986,
  'execution_mode': 'allocation',
  'pbs_job_id': '6874956.desched1',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_control_96f6d4898dc6.log',
  'action_count': 1,
  'action_trace': [{'action': {'count': 5, 'type': 'run_steps'},
    'index': 0,
    'last_phase': 'physics_timestep_initial',
    'last_scheme': 'physics_before_coupler.kessler_diagnostics',
    'last_scheme_group': 'physics_before_coupler',
    'native_calls_delta': 1490,
    'step_af

## 7. What the summary tells you

All three final branches report `step=15` and 16 history samples (the initial state plus 15 completed steps). `control` and `no-kessler` have `parent_branch='base'`; `warm` has `parent_branch='warm-initial'`. The summary also contains `execution_mode`, each PBS job ID, run/history/checkpoint/log paths, and serialized checkpoint size without downloading the full checkpoint Future. In allocation mode every PBS job ID must be identical.

In [7]:
[
    {
        'branch': name,
        'step': summary['step'],
        'history_samples': summary['history_samples'],
        'execution_mode': summary['execution_mode'],
        'pbs_job_id': summary['pbs_job_id'],
        'checkpoint_GiB': summary['snapshot_nbytes'] / 1024**3,
        'history_dir': summary['history_dir'],
        'log_path': summary['log_path'],
    }
    for name, summary in summaries.items()
]

[{'branch': 'control',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6874956.desched1',
  'checkpoint_GiB': 0.03507452644407749,
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/control/history',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_control_96f6d4898dc6.log'},
 {'branch': 'no-kessler',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6874956.desched1',
  'checkpoint_GiB': 0.03507457114756107,
  'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/no-kessler/history',
  'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_no-kessler_4684e0bb450d.log'},
 {'branch': 'warm',
  'step': 15,
  'history_samples': 16,
  'execution_mode': 'allocation',
  'pbs_job_id': '6874956.desched1',
  'checkpoint_GiB': 0.03507452644407749,
  'history_

## 8. Compare the temperature after five model steps

Every branch checkpoint contains the complete 214-field StatePool for all 24 ranks. Unlike the exact pre-run edit, the warm-control difference after five nonlinear model steps is expected to vary around `1 K`. The deviation from `1 K`, rather than the total warm-control difference, measures the subsequent model response.

In [8]:
control_temperature = checkpoint_field(
    summaries, 'control', 'air_temperature'
)
warm_temperature = checkpoint_field(
    summaries, 'warm', 'air_temperature'
)
temperature_difference = warm_temperature - control_temperature

{
    'rank': 0,
    'shape': control_temperature.shape,
    'difference_min': float(temperature_difference.min()),
    'difference_max': float(temperature_difference.max()),
    'difference_mean': float(temperature_difference.mean()),
    'maximum_deviation_from_1K': float(
        np.abs(temperature_difference - 1.0).max()
    ),
    'bitwise_identical': bool(np.array_equal(control_temperature, warm_temperature)),
}

{'rank': 0,
 'shape': (4, 4, 30, 3, 3),
 'difference_min': 0.9913538548482279,
 'difference_max': 1.0004294651569978,
 'difference_mean': 0.9997454265889733,
 'maximum_deviation_from_1K': 0.008646145151772089,
 'bitwise_identical': False}

## 9. Inspect a global field after every model step

Each branch history directory inherits the 10 base timestamps and adds 5 branch timestamps. These NetCDF files contain the 26 configured global diagnostics. Unlike the final rank-local checkpoint, this gives one global field value at every completed model step.

In [9]:
def history_statistics(branch, variable):
    summary = summaries[branch]
    history_dir = summary.get(
        'history_dir',
        Path(summary['checkpoint_dir']).parent / 'history',
    )
    files = sorted(Path(history_dir).glob('*.nc'))
    records = []
    for path in files:
        with Dataset(path) as dataset:
            values = np.asarray(dataset[variable][0])
            records.append({
                'step': int(dataset['nsteph'][0]),
                'file': path.name,
                'minimum': float(values.min()),
                'maximum': float(values.max()),
                'mean': float(values.mean()),
            })
    return records

control_rain_by_step = history_statistics('control', 'RAINQM')
no_kessler_rain_by_step = history_statistics('no-kessler', 'RAINQM')

{
    'control_last': control_rain_by_step[-1],
    'no_kessler_last': no_kessler_rain_by_step[-1],
    'control_all_steps': control_rain_by_step,
}

{'control_last': {'step': 15,
  'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-27000i.nc',
  'minimum': 0.0,
  'maximum': 0.0,
  'mean': 0.0},
 'no_kessler_last': {'step': 15,
  'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-27000i.nc',
  'minimum': 0.0,
  'maximum': 0.0,
  'mean': 0.0},
 'control_all_steps': [{'step': 0,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-00000i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 1,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-01800i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 2,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-03600i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 3,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-05400i.nc',
   'minimum': 0.0,
   'maximum': 0.0,
   'mean': 0.0},
  {'step': 4,
   'file': 'pycam_sima_legacy_oracle_f8daa568.cam.h0.0001-01-01-0720

## 10. Run phase and scheme actions

A `SegmentPlan` executes all actions inside one MPI segment, so its actions share one live StatePool without writing an intermediate checkpoint. `submit_action()` instead creates one new Dask task and one final checkpoint, which is useful when that action boundary must become a Future or fork point. Scheme actions use the same DeviceRegistry/CCPP-standard-name connection as the interactive model; Dask does not carry a second kernel implementation. Standalone phase and scheme calls are intentionally marked `unsafe=True`: they do not advance the model clock or automatically run prerequisite calculations.

In [10]:
granular_plan = SegmentPlan(
    'granular-kessler-then-map',
    actions=(
        RunScheme(
            'kessler',
            group='physics_before_coupler',
        ),
        ObserveFields((
            'potential_temperature',
            'large_scale_precipitation_rate',
        )),
        RunPhase('dynamics_to_physics'),
        ObserveFields(('air_temperature',)),
    ),
    unsafe=True,
)

granular = experiments.submit_plan(base, granular_plan)
granular_summary = experiments.summary(granular).result()
rank0_potential_temperature = experiments.field(
    granular,
    'potential_temperature',
    rank=0,
).result()

{
    'step': granular_summary['step'],
    'action_trace': granular_summary['action_trace'],
    'rank0_field_shape': rank0_potential_temperature.shape,
    'rank0_field_mean': float(rank0_potential_temperature.mean()),
}

{'step': 10,
 'action_trace': [{'action': {'group': 'physics_before_coupler',
    'name': 'kessler',
    'type': 'run_scheme'},
   'index': 0,
   'last_phase': 'physics_timestep_initial',
   'last_scheme': 'physics_before_coupler.kessler',
   'last_scheme_group': 'physics_before_coupler',
   'native_calls_delta': 1,
   'step_after': 10,
   'step_before': 10,
   'type': 'run_scheme'},
  {'action': {'fields': ['potential_temperature',
     'large_scale_precipitation_rate'],
    'statistics': ['min', 'max', 'mean'],
    'type': 'observe_fields'},
   'index': 1,
   'last_phase': 'physics_timestep_initial',
   'last_scheme': 'physics_before_coupler.kessler',
   'last_scheme_group': 'physics_before_coupler',
   'native_calls_delta': 0,
   'observations': [{'field': 'potential_temperature',
     'global': {'max': 757.6410760206371,
      'mean': 373.52033421106006,
      'min': 239.75108667553826},
     'ranks': [{'count': 810,
       'dtype': '<f8',
       'max': 748.2600683072949,
       'm

### Make one scheme boundary a separate Future

This form starts a new 24-rank MPI segment, restores `base`, runs exactly one scheme, and writes a checkpoint. A later `submit_action()`, `submit_plan()`, or `fork()` can use `kessler_only` as its parent.

In [11]:
kessler_only = experiments.submit_action(
    base,
    name='single-kessler-action',
    action=RunScheme(
        'kessler',
        group='physics_before_coupler',
    ),
)
experiments.summary(kessler_only).result()

{'branch': 'single-kessler-action',
 'parent_branch': 'base',
 'step': 10,
 'history_samples': 11,
 'run_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/single-kessler-action/run',
 'history_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/single-kessler-action/history',
 'checkpoint_dir': '/glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516/branches/single-kessler-action/checkpoint',
 'snapshot_nbytes': 37660698,
 'execution_mode': 'allocation',
 'pbs_job_id': '6874956.desched1',
 'log_path': '/glade/work/ruitong/pycam-sima/logs/pycam_dask_allocation_single-kessler-action_b7a7a3cf1ed0.log',
 'action_count': 1,
 'action_trace': [{'action': {'group': 'physics_before_coupler',
    'name': 'kessler',
    'type': 'run_scheme'},
   'index': 0,
   'last_phase': 'physics_timestep_initial',
   'last_scheme': 'physics_before_coupler.kessler',
   'last_sche

## 11. Keep one MPI model alive with a Dask Actor

The checkpoint examples above intentionally start a new MPI world for every Dask task. The persistent API below is different: Dask pins one Actor to one worker, the Actor starts `mpiexec -n 24` exactly once, and later Actor method calls reuse the same live StatePool and model clock. Each method returns an `ActorFuture`; `.result()` waits for that command. Close the Actor before submitting any more full-node checkpoint segments in the same allocation.

In [ ]:
persistent_model = experiments.start_persistent('persistent-live')
try:
    persistent_started = persistent_model.describe().result()
    persistent_step = persistent_model.step(2).result()
    persistent_temperature = persistent_model.field_stats(
        'air_temperature', rank=0
    ).result()
    persistent_checkpoint = persistent_model.checkpoint().result()
    persistent_final = persistent_model.describe().result()
finally:
    persistent_closed = persistent_model.close().result()

assert persistent_started['mpi_launch_count'] == 1
assert persistent_step['mpi_launch_count'] == 1
assert persistent_final['mpi_launch_count'] == 1
assert persistent_final['step'] == 2
{
    'worker': persistent_model.worker,
    'started': persistent_started,
    'rank0_temperature': persistent_temperature,
    'checkpoint': persistent_checkpoint,
    'closed': persistent_closed,
}

## 12. Fork three persistent models directly from memory

`fork_persistent()` captures the base StatePool as one immutable `CheckpointBundle` in Dask distributed memory. Every child starts its own 24-rank MPI model, receives the same bytes through its Actor/socket bridge, restores private NumPy arrays, applies its branch plan, and stays alive. No `rank-*.npz`, `manifest.json`, or checkpoint directory is created. This requires PBS mode and one Dask worker per concurrently controlled child; a one-node allocation cannot contain three independent 24-rank models.

In [ ]:
if execution_mode != 'pbs':
    persistent_fork_result = {
        'skipped': True,
        'reason': (
            'Set execution_mode="pbs" and create at least three Dask '
            'workers; one allocation node can host only one 24-rank model.'
        ),
    }
else:
    fork_base = experiments.start_persistent('memory-base')
    fork_base_closed = False
    persistent_children = {}
    try:
        fork_base.step(10).result()
        base_temperature = fork_base.field(
            'air_temperature', rank=0
        ).result()
        persistent_children = experiments.fork_persistent(
            fork_base,
            (
                BranchSpec('memory-control', steps=0),
                BranchSpec(
                    'memory-no-kessler',
                    steps=0,
                    disable_schemes=('kessler',),
                ),
                BranchSpec(
                    'memory-warm',
                    steps=0,
                    field_edits=(
                        FieldEdit('air_temperature', 'add', 1.0),
                    ),
                ),
            ),
            close_parent=True,
        )
        fork_base_closed = True
        initial_temperature = {
            name: child.field('air_temperature', rank=0).result()
            for name, child in persistent_children.items()
        }
        assert np.array_equal(
            initial_temperature['memory-control'], base_temperature
        )
        assert np.array_equal(
            initial_temperature['memory-no-kessler'], base_temperature
        )
        assert np.array_equal(
            initial_temperature['memory-warm'],
            np.add(base_temperature, 1.0),
        )
        step_futures = {
            name: child.step(1)
            for name, child in persistent_children.items()
        }
        persistent_fork_result = {
            name: {
                'status': child.describe().result(),
                'after_step': step_futures[name].result(),
            }
            for name, child in persistent_children.items()
        }
        memory_branch_roots = (
            run_root / 'memory-control',
            run_root / 'memory-no-kessler',
            run_root / 'memory-warm',
        )
        assert not any(
            tuple(root.glob('**/checkpoints'))
            or tuple(root.glob('**/*.npz'))
            for root in memory_branch_roots
        )
    finally:
        for child in persistent_children.values():
            child.close().result()
        if not fork_base_closed:
            fork_base.close().result()

persistent_fork_result

In [12]:
client.close()
print('Dask client closed; PBS results remain under', experiment_root)

Dask client closed; PBS results remain under /glade/derecho/scratch/ruitong/pycam-sima/dask_notebook_trials/fanout-20260723-182516


In [13]:
import os
import socket
import sys

print("host:", socket.gethostname())
print("python:", sys.executable)
print("PBS_JOBID:", os.environ.get("PBS_JOBID"))
print("PBS_NODEFILE:", os.environ.get("PBS_NODEFILE"))

host: dec1195
python: /glade/work/ruitong/pycam-sima/.venv/bin/python
PBS_JOBID: 6874956.desched1
PBS_NODEFILE: /var/spool/pbs/aux/6874956.desched1
